# Automated Clickbait Headline Generator — Demo

This notebook demonstrates the end-to-end pipeline:
1. Load a trained model checkpoint
2. Generate headlines with various seeds and temperatures
3. Visualise the training loss / perplexity curves


In [ ]:
import sys
import os

# Add src/ to the path so we can import our modules
sys.path.insert(0, os.path.join('..', 'src'))

import json
import math
import torch
import matplotlib.pyplot as plt

from generate import load_model, generate_headline, generate_batch

CHECKPOINT = '../checkpoints/best_model.pt'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Load Trained Model

In [ ]:
model, word2idx, idx2word = load_model(CHECKPOINT, device)
print(f'Vocabulary size: {len(word2idx)}')
print(f'Model: {type(model).__name__}')

## 2. Generate Headlines — Unconditional

In [ ]:
print('=== Unconditional Generation (temperature=1.0) ===')
for h in generate_batch(model, word2idx, idx2word, seed_phrase='', num_headlines=5,
                         temperature=1.0, device=device):
    print(f'  • {h}')

## 3. Generate Headlines — With Seed Phrase

In [ ]:
seeds = ['10 things', 'you won\'t believe', 'the real reason', 'this is why']

for seed in seeds:
    print(f'\n=== Seed: "{seed}" ===')
    headlines = generate_batch(model, word2idx, idx2word,
                               seed_phrase=seed, num_headlines=3,
                               temperature=0.8, device=device)
    for h in headlines:
        print(f'  • {h}')

## 4. Effect of Temperature

In [ ]:
seed = '10 things'
for temp in [0.4, 0.7, 1.0, 1.3]:
    h = generate_headline(model, word2idx, idx2word,
                          seed_phrase=seed, temperature=temp, device=device)
    print(f'  T={temp:.1f}  →  {h}')

## 5. Training Loss & Perplexity Curves

In [ ]:
history_path = '../checkpoints/history.json'

with open(history_path) as f:
    history = json.load(f)

train_losses = history['train_loss']
val_losses   = history['val_loss']
epochs = range(1, len(train_losses) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, train_losses, label='Train Loss')
axes[0].plot(epochs, val_losses,   label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True)

train_ppl = [math.exp(l) for l in train_losses]
val_ppl   = [math.exp(l) for l in val_losses]
axes[1].plot(epochs, train_ppl, label='Train Perplexity')
axes[1].plot(epochs, val_ppl,   label='Val Perplexity')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Perplexity')
axes[1].set_title('Training and Validation Perplexity')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()